## **Evaluating ConvNeXt-tiny on the Oxford Flowers102 dataset**

First, we import all the necessary libraries and functions needed to run the code.

Second, we setup the device and check if cuda is available to speed up code performance.

Third, we define all the data preprocessing and normalization, then import and load the dataset.

Fourth, we load the predefined ConvNeXt-t model with its weights and setup the loss function, optimizer, and scheduler etc.

Fifth, we check if there was a checkpoint for the training, then we start the training loop and validate the training to save the best model.

Lastly, we evaluate the best trained model on the testing dataset to get the accuracies.

In [9]:
# Imports and environment setup
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torchvision.models import convnext_tiny, ConvNeXt_Tiny_Weights
from torch.utils.data import DataLoader, random_split, ConcatDataset
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
import torch.optim as optim
from tqdm import tqdm
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
import numpy as np
import gc
import os

In [10]:
# Device setup and check GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.backends.cudnn.benchmark = True
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
    print("CUDA is available! Training on GPU...")
else:
    print("CUDA is not available. Training on CPU...")

NVIDIA GeForce RTX 2080 SUPER
CUDA is available! Training on GPU...


In [11]:
# Train transform with data augmentation
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandAugment(),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(30),
    transforms.RandomResizedCrop(224, scale=(0.7, 1.3)),
    transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5, hue=0.2),
    transforms.RandomAffine(degrees=15, translate=(0.1, 0.1), scale=(0.8, 1.3), shear=10),
    transforms.RandomPerspective(distortion_scale=0.5, p=0.5),
    transforms.RandomErasing(p=0.5, scale=(0.03, 0.33), ratio=(0.3, 3.3)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# Validation transform with only normalization
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# Load train dataset
train_dataset = torchvision.datasets.Flowers102(
    root='./data',
    split='test',
    download=True,
    transform=train_transform
)

# Load val dataset
val_dataset = torchvision.datasets.Flowers102(
    root='./data',
    split='val',
    download=True,
    transform=val_transform
)

# Load test dataset
test_dataset = torchvision.datasets.Flowers102(
    root='./data',
    split='train',
    download=True,
    transform=val_transform
)

# Concatenate the validation and testing datasets
test_dataset = ConcatDataset([val_dataset, test_dataset])

# Split into train and val
train_size = int(0.75 * len(train_dataset))
val_size = len(train_dataset) - train_size
train_dataset, val_dataset = random_split(train_dataset, [train_size, val_size])

# Update val dataset transform
val_dataset.dataset.transform = val_transform

# Create dataloaders
batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=True)

# Print the size of the split datasets
print(f"Training sample: {len(train_dataset)}. Size: {len(train_loader)}")
print(f"Validation sample: {len(val_dataset)}. Size: {len(val_loader)}")
print(f"Testing sample: {len(test_dataset)}. Size: {len(test_loader)}")

Training sample: 4611. Size: 73
Validation sample: 1538. Size: 25
Testing sample: 2040. Size: 32


In [12]:
# Load pretrained ConvNeXt-Tiny model
weights = ConvNeXt_Tiny_Weights.IMAGENET1K_V1
model = convnext_tiny(weights=weights)
model.classifier.insert(1, nn.Dropout(0.4))
model.classifier[3] = nn.Linear(model.classifier[3].in_features, 102)  # 102 classes for Flowers102
model = model.to(device)

# Setup loss function, Optimizer and scheduler
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.1)
scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2)
scaler = torch.amp.GradScaler('cuda')

# Training setup
num_epochs = 50
best_val_acc = 0
patience = 10
patience_counter = 0

In [13]:
# Load checkpoint if exists
checkpoint_path = 'checkpoint_convnext_flower.pth'
if os.path.exists(checkpoint_path):
    checkpoint = torch.load(checkpoint_path)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scaler.load_state_dict(checkpoint['scaler_state_dict']) 
    start_epoch = checkpoint['epoch'] + 1
    best_val_acc = checkpoint['best_val_acc']
    patience_counter = checkpoint['patience_counter']
    print(f"Resumed training from epoch {start_epoch}\n")
else:
    start_epoch, best_val_acc, patience_counter = 0, 0, 0
    print(f"Starting training from scratch\n")

# Training loop
for epoch in range(start_epoch, num_epochs):
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    # Clear cache
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

    # Training
    for inputs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}"):
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()

        # Forward pass with automatic mixed precision (AMP) and use autocast for mixed precision training
        with torch.amp.autocast('cuda'):
            outputs = model(inputs)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_loss = running_loss / len(train_loader)
    train_acc = correct / total

    # Validation
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            val_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    val_loss /= len(val_loader)
    val_acc = val_correct / val_total

    scheduler.step()

    print(f"Epoch [{epoch+1}/{num_epochs}] Train Loss: {train_loss:.4f} Train Acc: {train_acc:.4f} Val Loss: {val_loss:.4f} Val Acc: {val_acc:.4f}")

    # Save the model checkpoint
    torch.save({
    'epoch': epoch,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'scaler_state_dict': scaler.state_dict(),  # if using AMP
    'best_val_acc': best_val_acc,
    'patience_counter': patience_counter
    }, 'checkpoint_convnext_flower.pth')
    
    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_counter = 0
        torch.save(model.state_dict(), 'best_convnext_model_flower.pth')
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            break

# Clear cache
if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()

Resumed training from epoch 45



Epoch 46/50: 100%|██████████| 73/73 [00:37<00:00,  1.94it/s]


Epoch [46/50] Train Loss: 0.7925 Train Acc: 1.0000 Val Loss: 0.7848 Val Acc: 0.9993


Epoch 47/50: 100%|██████████| 73/73 [00:36<00:00,  1.99it/s]


Epoch [47/50] Train Loss: 0.7918 Train Acc: 1.0000 Val Loss: 0.7856 Val Acc: 0.9993


Epoch 48/50: 100%|██████████| 73/73 [00:36<00:00,  2.00it/s]


Epoch [48/50] Train Loss: 0.7923 Train Acc: 0.9998 Val Loss: 0.7854 Val Acc: 0.9993
Early stopping at epoch 48


743

In [15]:
# Load best model for testing
model.load_state_dict(torch.load('best_convnext_model_flower.pth'))
model.eval()

# Test evaluation
test_loss, test_correct, test_total = 0.0, 0, 0
all_preds = []
all_labels = []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        loss = criterion(outputs, labels)

        test_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        test_total += labels.size(0)
        test_correct += (predicted == labels).sum().item()
        
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

test_loss /= len(test_loader)
test_acc = test_correct / test_total

print(f"Test Loss: {test_loss:.4f} Test Acc: {test_acc:.4f}")

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

# Calculate metrics
top1_acc = accuracy_score(all_labels, all_preds)
precision = precision_score(all_labels, all_preds, average='macro')
recall = recall_score(all_labels, all_preds, average='macro')
f1 = f1_score(all_labels, all_preds, average='macro')

print(f"\nTop-1 Accuracy: {top1_acc:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-Score: {f1:.4f}")

Test Loss: 0.8512 Test Acc: 0.9809

Top-1 Accuracy: 0.9809
Precision: 0.9818
Recall: 0.9809
F1-Score: 0.9808
